### Setup and Installation
This cell installs the necessary Python libraries for building a Retrieval-Augmented Generation (RAG) pipeline using LlamaIndex. It includes components for language models (LLMs), embeddings, retrievers (BM25), document readers, and asynchronous operations.

In [ ]:
!pip -q install -U \
  llama-index \
  llama-index-llms-openai \
  llama-index-embeddings-openai \
  llama-index-embeddings-huggingface \
  llama-index-llms-llama-cpp \
  llama-index-retrievers-bm25 \
  llama-index-readers-file \
  nest-asyncio

In [ ]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    print("GPU is available, using GPU.")
    device = "cuda"
else:
    print("GPU not available, using CPU.")
    device = "cpu"

!pip -q install -U sentence-transformers

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Using `BAAI/bge-small-en-v1.5` as the local embedding model, which is a good balance between performance and size.

In [ ]:
from llama_index.core import Settings

# Set the embedding model to a local HuggingFace model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device=device
)

### Asynchronous Operations Fix
`nest_asyncio` is imported and `nest_asyncio.apply()` is called to allow nested execution of asyncio event loops.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

### Create Knowledge Base Directory
`os.makedirs` is used to create a directory named `tech_kb`. The `exist_ok=True` argument ensures that if the directory already exists, no error is raised.

In [ ]:
import os
os.makedirs("tech_kb", exist_ok=True)

### Define Knowledge Base Content
This dictionary, `docs`, stores the content for our knowledge base.

In [ ]:
docs = {
    "rag_basics.txt": """Retrieval-Augmented Generation, or RAG, combines retrieval with language generation.\nA typical RAG pipeline retrieves relevant chunks and uses them as context for the LLM.\nRAG reduces hallucination by grounding answers in external knowledge.\nHowever, vector-only retrieval may miss results when exact keywords matter or when the query has multiple parts.""",

    "hybrid_search.txt": """Hybrid retrieval combines dense vector retrieval and sparse BM25 retrieval.\nDense retrieval captures semantic similarity between query and text.\nBM25 captures keyword overlap and term importance.\nReciprocal Rank Fusion combines ranked lists from multiple retrievers to improve retrieval quality.""",

    "reranking_note.txt": """In production systems, a second-stage reranker may be used after retrieval.\nIn this lab, we use reciprocal rank fusion as the ranking strategy, so no separate reranker API is required.""",

    "query_decomposition.txt": """Query decomposition breaks one complex user question into smaller sub-queries.\nThis improves recall because each sub-query can retrieve useful evidence.\nFusion-based retrieval systems often combine query decomposition with multi-retriever search.""",

    "citations.txt": """Citations show which source documents support the generated answer.\nThey improve trust, transparency, and verification in a RAG system.\nLlamaIndex can return source nodes along with the final response."""
}

### Write Content to Files
This cell iterates through the `docs` dictionary and writes content to files.

In [ ]:
import textwrap
for filename, content in docs.items():
    with open(os.path.join("tech_kb", filename), "w", encoding="utf-8") as f:
        f.write(textwrap.dedent(content).strip())

### Load Documents
`SimpleDirectoryReader` from LlamaIndex is used to load all text documents from the `tech_kb` directory.

In [ ]:
from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader("tech_kb").load_data()
print("Documents loaded:", len(documents))

### Configure LlamaIndex Settings
This cell configures core components of LlamaIndex.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=40)

### Create Vector Store Index
`VectorStoreIndex.from_documents` creates an index from the loaded documents.

In [ ]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)
print("Vector index ready")

### Initialize Vector Retriever
This cell converts the `vector_index` into a `vector_retriever`.

In [ ]:
vector_retriever = vector_index.as_retriever(similarity_top_k=4)

### Initialize BM25 Retriever
This cell sets up a `BM25Retriever` for keyword-based retrieval.

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core import Settings

nodes = Settings.node_parser.get_nodes_from_documents(documents)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=4)

### Create Query Fusion Retriever
`QueryFusionRetriever` combines results from multiple retrievers using reciprocal rank fusion.

In [ ]:
from llama_index.core.retrievers import QueryFusionRetriever

fusion_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=6,
    num_queries=4,
    mode="reciprocal_rerank",
    use_async=True,
    verbose=True,
)

print("Hybrid fusion retriever created")

### Configure Response Synthesizer
`get_response_synthesizer` creates an object responsible for generating the final answer.

In [ ]:
from llama_index.core.response_synthesizers import get_response_synthesizer

response_synthesizer = get_response_synthesizer(response_mode="compact")

### Create Retriever Query Engine
This cell initializes the `RetrieverQueryEngine`, the core component of our RAG pipeline.

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine(
    retriever=fusion_retriever,
    response_synthesizer=response_synthesizer,
)

print("Advanced RAG pipeline ready")

### Define the Query
This cell defines the query string for the RAG pipeline.

In [ ]:
query = """
Explain how query decomposition, hybrid retrieval, and reciprocal rank fusion
work together in an advanced RAG pipeline. Also mention why citations matter.
"""

### Execute the Query
This cell executes the query using the query_engine.

In [ ]:
response = query_engine.query(query)
print("Response:")
print(response)